# Deep Agents 101: Journal Agent

This notebook builds one agent, one step at a time. Slides cover the concepts (harness, tools, HITL); this notebook covers the implementation.

For topics not covered today, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langchain-anthropic langgraph tavily-python dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Requires ANTHROPIC_API_KEY in your .env file

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="anthropic:claude-sonnet-4-5",
    api_key=os.environ["ANTHROPIC_API_KEY"],
)

# To use a different model instead, replace the block above, for example:
# model = init_chat_model("anthropic:claude-haiku-4-5", api_key=os.environ["ANTHROPIC_API_KEY"])
# model = init_chat_model("openai:gpt-4.1", api_key=os.environ["OPENAI_API_KEY"])

## 1: The harness, what you get before writing any tool code

Every deep agent starts the same way: a model wrapped in a harness that already knows how to...
- read and write files
- plan, and call tools

No tools are added yet. The next cell shows what it can already do.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model
    )

message = """
    Start a journal.md file. Log a new dated entry from these notes:
    - What I learned today: a caching bug was hiding in the retry logic
    - How I felt: relieved to ship it, a little anxious about production traffic
    - What's next: write better tests before touching anything else
    Then read the file back to me.
    """

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

## 2: Set its role with a system prompt

One `system_prompt` string controls how the agent behaves, on top of whatever instructions/facts you give it.

In [ ]:
# Try a different persona by uncommenting one of these (or write your own):
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a toddler. Explain everything like you're five."
system_prompt = "You are a melodramatic Victorian child. Narrate everything with excessive despair and flowery, dramatic language."

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt
    )

message = """
    Log a three-sentence journal entry from these notes:
    - what I learned today (a caching bug was hiding in the retry logic)
    - how I felt (relieved but a little anxious)
    - what's next (write better tests before touching anything else).
    """

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

## 3: Give it a custom tool

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back

In [ ]:
from langchain_core.tools import tool

@tool
def word_count(text: str) -> str:
    """Count the words in a piece of text."""

    words = len(text.split())

    return f"{words} words"

In [ ]:
tools = [word_count]

system_prompt = "Count the words in the following journal entry using your tool and tell me the result."

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=tools
    )

message = """
    Today I finally shipped the feature I've been stuck on for a week.
    The bug turned out to be a caching issue that took forever to track down, and I
    ended up rewriting most of the retry logic to fix it. It feels good to have it done,
    though I'm a little worried about whether the fix will hold up under real traffic.
    Tomorrow I want to write better tests before touching anything else.
    """

result = agent.invoke({"messages": [{"role": "user", "content": message}]})

for m in result['messages']:
    m.pretty_print()

## Short-term memory, so you can respond to your agent

Short-term memory with any agent built using any LangChain agent framework, including Deep Agents, is managed using a `checkpointer`.

In [ ]:
# Import the checkpointer

from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
# The following thread_id is what tracks the conversation over turns, change the thread_id to change the conversation

config = {"configurable": {"thread_id": "1"}}

In [ ]:
# Create an agent using a checkpointer
agent = create_deep_agent(
    model=model,
    checkpointer=checkpointer
    )

# Invoke agent with the thread_id (config) to track the conversation between turns
message = "Hello my name is Jess"

result = agent.invoke({"messages": [{"role": "user", "content": message}]},
                      config=config
                      )

for m in result['messages']:
    m.pretty_print()

In [ ]:
# Send a follow up message
message = "What's my name?"

result = agent.invoke({"messages": [{"role": "user", "content": message}]},
                      config=config
                      )

for m in result['messages']:
    m.pretty_print()

## 4: Human-in-the-loop, approve a risky action before it happens

`interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes.

Resume by calling `agent.invoke` again with

`Command(resume={"decisions": [{"type": "approve"}]})`

(or `"edit"` / `"reject"`)

rather than starting a new conversation from scratch.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def share_journal_entry(entry: str, platform: str) -> str:
    """Share a journal entry to an external platform. This just simulates a send, no network call is actually made."""
    return f"Shared to {platform}: {entry[:60]}..."

tools = [share_journal_entry]

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=model,
    tools=tools,
    interrupt_on={"share_journal_entry": True},
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-hitl-demo"}}

message = """
        Start a journal.md file. Log a new dated entry:
        'Set up human-in-the-loop approval today, it feels reassuring to have a real gate before anything gets shared externally.'
        Then read the file back, and share the most recent entry to the 'team-standup' platform.
"""

result = agent.invoke(
    {"messages": [{"role": "user", "content": message}]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    print(result["messages"][-1].content)

The cell above paused instead of finishing, because `share_journal_entry` matched `interrupt_on`. The cell below resumes it with an approval decision.

In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)

for m in result['messages']:
    m.pretty_print()

## Wrap-up

You built: a filesystem-backed agent, a way to swap personas, and a custom tool.

Also covered: human-in-the-loop gating, added with a single argument.

Not covered today, but in the full LangChain Academy Deep Agents course: subagent delegation, backends (filesystem/store/composite), skills, memory across sessions, sandboxes, and deployment.